---
# FASE 5 | EVALUACION (Evaluation)
---

## 5.1 Metricas en el conjunto de prueba

In [ ]:
test_results = evaluate_on_test(trained_models, X_test_proc, y_test.values)

In [ ]:
print('\nTabla de Metricas - Conjunto de Prueba:')
display(test_results.round(4))

In [ ]:
plot_metrics_heatmap(test_results, save_path=f'{FIGURES_DIR}/metrics_heatmap.png')

## 5.2 Matrices de confusión

In [ ]:
plot_confusion_matrices(trained_models, X_test_proc, y_test.values,
                         save_path=f'{FIGURES_DIR}/confusion_matrices.png')

## 5.3 Curvas ROC

In [ ]:
plot_roc_curves(trained_models, X_test_proc, y_test.values,
                save_path=f'{FIGURES_DIR}/roc_curves.png')

## 5.4 Importancia de caracteristicas

In [ ]:
for name, model in trained_models.items():
    plot_feature_importance(
        model, feature_names_proc, name, top_n=15,
        save_path=f'{FIGURES_DIR}/feature_importance.png'
    )

## 5.5 Comparación global y selección del mejor modelo

In [ ]:
# Comparacion: CV vs Test
summary = pd.DataFrame({
    'CV ROC-AUC': cv_results['ROC-AUC (val)'],
    'Test ROC-AUC': test_results['ROC-AUC'],
    'CV F1': cv_results['F1 (val)'],
    'Test F1': test_results['F1'],
    'Test Accuracy': test_results['Accuracy'],
})

print('Comparacion CV vs Conjunto de Prueba:')
display(summary.round(4))

mejor_modelo = test_results['ROC-AUC'].idxmax()
mejor_auc    = test_results.loc[mejor_modelo, 'ROC-AUC']

print(f'\nMejor modelo (por ROC-AUC en prueba): {mejor_modelo}')
print(f'ROC-AUC = {mejor_auc:.4f}')

In [ ]:
# Grafica de barras lado a lado: CV vs Test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

model_colors = ['#4C72B0', '#55A868', '#C44E52']

for ax, metric in zip(axes, ['ROC-AUC', 'F1']):
    cv_vals   = summary[f'CV {metric}'].values
    test_vals = summary[f'Test {metric}'].values
    x = np.arange(len(summary))
    w = 0.35

    bars1 = ax.bar(x - w/2, cv_vals,   w, label='Validacion Cruzada',
                   color=model_colors, alpha=0.6)
    bars2 = ax.bar(x + w/2, test_vals, w, label='Conjunto de Prueba',
                   color=model_colors, alpha=0.95)

    ax.set_xticks(x)
    ax.set_xticklabels(summary.index, rotation=12)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.1)
    ax.set_title(f'{metric}: CV vs Prueba', fontweight='bold')
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)

    for bar, val in zip(bars2, test_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Comparacion Final: Validacion Cruzada vs Conjunto de Prueba',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/comparacion_final.png', bbox_inches='tight')
plt.show()